# Electrode Positions on a Scalp Topomap

**Dataset**: BNCI2014-001 (Motor Imagery)
**Subject**: 1
**Channels**: 22 EEG
**Sampling rate**: 250 Hz

---

## Overview

This notebook loads the BNCI2014-001 dataset, extracts the 22 EEG channel positions from the raw MNE info, and visualizes the electrode layout on a 2D scalp topomap with a head circle and labeled electrode dots.

## What this notebook does

- Loads BNCI2014-001 subject 1 via MOABB
- Extracts channel positions from the raw montage
- Projects 3D positions to 2D (top view)
- Plots electrodes as labeled dots inside a head circle

## What you should expect to see

- A circular head outline with a nose marker at the top
- 22 labeled electrode dots (Fz, FC3, C3, Cz, C4, Pz, etc.)
- Standard 10-20 extended layout covering frontal, central, and parietal regions
- Motor cortex electrodes (C3, C1, Cz, C2, C4) in the center

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| dataset | BNCI2014_001 | MOABB motor imagery dataset |
| subjects | [1] | Subject 1 only |
| n_electrodes | 22 | EEG electrodes with positions |
| projection | top | 2D top-view projection |



## 1. Install dependencies


In [ ]:
!pip install moabb mne scipy numpy plotly


## 2. Load MOABB dataset

MOABB downloads data automatically on first use (~44 MB for subject 1). Subsequent runs use cached data.



In [ ]:
from moabb.datasets import BNCI2014_001
ds = BNCI2014_001()
sessions = ds.get_data(subjects=[1])
subject_key = list(sessions.keys())[0]
session_dict = sessions[subject_key]
n_sessions = len(session_dict)
n_runs = len(next(iter(session_dict.values())))
first_run = next(iter(next(iter(session_dict.values())).values()))
n_channels_raw = len(first_run.ch_names)
sfreq = first_run.info['sfreq']
print(f'Subject 1: {n_sessions} sessions, {n_runs} runs/session')
print(f'Raw channels: {n_channels_raw}, Sampling rate: {sfreq} Hz')
print(f'Channel names: {first_run.ch_names}')



In [ ]:
from moabb.paradigms import MotorImagery
paradigm = MotorImagery(n_classes=2)
X, labels, meta = paradigm.get_data(dataset=ds, subjects=[1])
print(f'X shape: {X.shape}  (n_trials, n_channels, n_samples)')
print(f'Labels shape: {labels.shape}')
print(f'Meta shape: {meta.shape}')



## 3. Explore the data

We print the channel names and sampling rate.


In [ ]:
import numpy as np
unique_labels, counts = np.unique(labels, return_counts=True)
print(f'Epoch channels: {X.shape[1]}')
print(f'Epoch samples: {X.shape[2]}')
print(f'Epoch duration: {X.shape[2] / sfreq:.2f} s')
print(f'Unique labels: {list(unique_labels)}')
print(f'Trials per class: {dict(zip(unique_labels, counts))}')
print(f'Total trials: {X.shape[0]}')



## 4. Apply the analysis

We extract the 3D channel positions and project them to 2D for the topomap.



In [ ]:
import numpy as np
montage = first_run.get_montage()
ch_pos = montage.get_positions()['ch_pos']
ch_names = [ch for ch in first_run.ch_names if ch in ch_pos]
pos = np.array([ch_pos[ch] for ch in ch_names])
pos_2d = pos[:, :2]
scale = 1.0 / np.max(np.abs(pos_2d))
pos_2d = pos_2d * scale * 0.95
print(f'Electrodes with positions: {len(ch_names)}')
print(f'Channel names: {ch_names}')



## 5. Interactive plot

**What to look for:**

- The head is represented as a circle with a nose marker at top
- 22 electrodes are labeled and positioned according to the 10-20 system
- Central electrodes (C3, C1, Cz, C2, C4) are key for motor imagery
- Frontal (Fz) and parietal (Pz) electrodes frame the layout




In [ ]:
import plotly.graph_objects as go
theta = np.linspace(0, 2*np.pi, 100)
head_x = np.cos(theta)
head_y = np.sin(theta)
fig = go.Figure()
fig.add_trace(go.Scatter(x=head_x, y=head_y, mode='lines', line=dict(color='black', width=2),
                         showlegend=False))
fig.add_trace(go.Scatter(x=[0, -0.06, 0.06], y=[1.0, 1.1, 1.1], fill='toself',
                         mode='lines', line=dict(color='black', width=1), showlegend=False))
fig.add_trace(go.Scatter(x=pos_2d[:, 0], y=pos_2d[:, 1], mode='markers+text',
                         text=ch_names, textposition='middle center',
                         marker=dict(size=22, color='#1f77b4', line=dict(color='black', width=1)),
                         textfont=dict(size=8, color='white'), showlegend=False))
fig.update_layout(title='Electrode Positions - BNCI2014-001 (22 channels)',
                  xaxis=dict(scaleanchor='y', scaleratio=1, range=[-1.2, 1.2], showgrid=False, zeroline=False, showticklabels=False),
                  yaxis=dict(range=[-1.2, 1.2], showgrid=False, zeroline=False, showticklabels=False),
                  width=700, height=700, plot_bgcolor='white')
fig.show()



## What did we learn?

- MOABB datasets include standard electrode montages with 3D positions
- The 22-channel layout follows the 10-20 extended system
- Electrode positions are essential for spatial analysis and topographic plotting
- Motor imagery BCI relies on central electrodes (C3, Cz, C4)


